# 02 - BGE 嵌入设置

本 notebook 直接从 ModelScope 下载并使用 `BAAI/bge-small-en-v1.5` 嵌入模型。

我们将:
- 将源文档下载到项目本地
- 将本地文件分割成块
- 使用 `modelscope.snapshot_download` 下载模型
- 从下载的本地路径初始化 LangChain 嵌入
- 创建 FAISS 向量存储
- 将模型保存在 `data/models/` 下
- 将向量存储保存到 `data/vector_stores/huggingface_embeddings`

**前置条件:**
- 已完成 [01_setup_and_basics.ipynb](01_setup_and_basics.ipynb)
- 已安装 `modelscope`

**预计时长:** ~5 分钟

## 1. 设置

将源文档下载到项目中,然后从本地加载它们并分割成块。

In [13]:
import sys
sys.path.append('../..')

import datetime
import os
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from langchain_core.documents import Document

from shared.config import (
    DEFAULT_CHUNK_OVERLAP,
    DEFAULT_CHUNK_SIZE,
    DEFAULT_LANGCHAIN_URLS,
    HF_VECTOR_STORE_PATH,
    PROJECT_ROOT,
    USER_AGENT,
)
from shared.loaders import split_documents
from shared.utils import print_section_header, save_vector_store

print_section_header("加载文档和块")

LOCAL_DOCS_DIR = PROJECT_ROOT / 'data' / 'source_docs' / 'langchain'
LOCAL_DOCS_DIR.mkdir(parents=True, exist_ok=True)

print(f"正在下载 {len(DEFAULT_LANGCHAIN_URLS)} 个文档到: {LOCAL_DOCS_DIR}")

docs = []
current_date = datetime.date.today().isoformat()

for index, url in enumerate(DEFAULT_LANGCHAIN_URLS, start=1):
    slug = url.rstrip('/').split('/')[-1] or f'doc_{index}'
    local_path = LOCAL_DOCS_DIR / f'{index:02d}_{slug}.txt'

    print(f"  - 正在下载 {url}")
    response = requests.get(url, headers={'User-Agent': USER_AGENT}, timeout=30)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'html.parser')
    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()

    lines = [line.strip() for line in soup.get_text('\n').splitlines()]
    text = '\n'.join(line for line in lines if line)
    local_path.write_text(text, encoding='utf-8')

    docs.append(
        Document(
            page_content=text,
            metadata={
                'source': url,
                'local_path': str(local_path),
                'source_type': 'local_text_download',
                'process_date': current_date,
                'domain': 'langchain',
            },
        )
    )

print(f"已下载 {len(docs)} 个文档")

chunks = split_documents(
    docs,
    chunk_size=DEFAULT_CHUNK_SIZE,
    chunk_overlap=DEFAULT_CHUNK_OVERLAP,
    verbose=True,
)

print(f"\n准备就绪: {len(docs)} 个文档和 {len(chunks)} 个块")


加载文档和块

正在下载 4 个文档到: d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..\data\source_docs\langchain
  - 正在下载 https://python.langchain.com/docs/use_cases/question_answering/
  - 正在下载 https://python.langchain.com/docs/modules/data_connection/retrievers/
  - 正在下载 https://python.langchain.com/docs/modules/model_io/llms/
  - 正在下载 https://python.langchain.com/docs/use_cases/chatbots/
已下载 4 个文档
Splitting documents...
  - Chunk size: 1000
  - Chunk overlap: 200
✓ Created 149 chunks

  Sample chunk:
    - Length: 981 chars
    - Source: https://python.langchain.com/docs/use_cases/question_answering/
    - Preview: Build a RAG agent with LangChain - Docs by LangChain
Skip to main content
Join us May 13th & May 14th at Interrupt, the Agent Conference by LangChain....

准备就绪: 4 个文档和 149 个块


## 2. 解析 BGE 模型

如果存在则复用本地项目副本;否则将模型下载到项目目录中。

In [14]:
from modelscope import snapshot_download

print_section_header("解析 BGE 模型")

MODEL_DIR = PROJECT_ROOT / 'data' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"模型根目录: {MODEL_DIR}")

existing_model_dirs = sorted(
    path.parent
    for path in MODEL_DIR.rglob('modules.json')
    if path.parent.name.startswith('bge-small-en-v1')
)

if existing_model_dirs:
    model_dir = str(existing_model_dirs[0])
    print("找到现有的本地模型目录")
    print(f"模型目录: {model_dir}")
else:
    print("未找到本地模型,正在从 ModelScope 下载...")
    start_time = time.time()
    model_dir = snapshot_download(
        'BAAI/bge-small-en-v1.5',
        cache_dir=str(MODEL_DIR),
    )
    elapsed_download = time.time() - start_time
    print("下载完成")
    print(f"模型目录: {model_dir}")
    print(f"耗时: {elapsed_download:.2f}s")


解析 BGE 模型

模型根目录: d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..\data\models
找到现有的本地模型目录
模型目录: d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..\data\models\BAAI\bge-small-en-v1___5


## 3. 初始化嵌入

从下载的本地项目路径加载嵌入模型,并使用示例查询进行验证。

In [15]:
from shared.config import DirectSentenceTransformerEmbeddings

print_section_header("初始化 BGE 嵌入")

print("正在从本地 ModelScope 路径初始化嵌入...")
bge_embeddings = DirectSentenceTransformerEmbeddings(model_name=model_dir)
print("BGE 嵌入已初始化")

test_query = "什么是检索增强生成?"
print(f"\n测试查询: '{test_query}'")

start_time = time.time()
test_embedding = bge_embeddings.embed_query(test_query)
elapsed_embedding = time.time() - start_time

print("\n结果:")
print(f"  维度: {len(test_embedding)}")
print(f"  时间: {elapsed_embedding:.3f}s")
print(f"  前 5 个值: {[f'{v:.4f}' for v in test_embedding[:5]]}")


初始化 BGE 嵌入

正在从本地 ModelScope 路径初始化嵌入...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BGE 嵌入已初始化

测试查询: '什么是检索增强生成?'

结果:
  维度: 384
  时间: 0.032s
  前 5 个值: ['-0.0178', '-0.0176', '-0.0170', '-0.0312', '0.0082']


## 4. 创建向量存储

使用下载的 BGE 嵌入模型构建 FAISS 向量存储。

In [16]:
from langchain_community.vectorstores import FAISS

print_section_header("创建向量存储")

print("正在使用 BGE 嵌入创建 FAISS 向量存储...")
start_time = time.time()
vectorstore_hf = FAISS.from_documents(chunks, bge_embeddings)
elapsed_vectorstore = time.time() - start_time

print(f"BGE 向量存储在 {elapsed_vectorstore:.2f}s 内创建")
print(f"  - 已索引 {len(chunks)} 个文档")
print(f"  - 嵌入维度: {len(test_embedding)}")
print("  - 模型: BAAI/bge-small-en-v1.5")


创建向量存储

正在使用 BGE 嵌入创建 FAISS 向量存储...


`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


BGE 向量存储在 11.12s 内创建
  - 已索引 149 个文档
  - 嵌入维度: 384
  - 模型: BAAI/bge-small-en-v1.5


## 5. 保存向量存储

持久化向量存储以避免在后续 notebook 中重新嵌入。

In [17]:
print_section_header("保存向量存储")

save_vector_store(vectorstore_hf, HF_VECTOR_STORE_PATH, verbose=True)

print("\n向量存储已成功保存")
print(f"位置: {HF_VECTOR_STORE_PATH}")


保存向量存储

Saved vector store to d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..\data\vector_stores\huggingface_embeddings

向量存储已成功保存
位置: d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\fundamentals\..\..\data\vector_stores\huggingface_embeddings


## 6. 测试相似度搜索

对保存的向量存储运行快速检索检查。

In [ ]:
from shared.utils import print_results

print_section_header("测试相似度搜索")

query = "How to build a RAG agent with LangChain?"
print(f"查询: '{query}'\n")

results_bge = vectorstore_hf.similarity_search(query, k=3)
print_results(results_bge, max_docs=3, preview_length=150)

print("\n相似度搜索成功完成")


测试相似度搜索

查询: '如何使用 LangChain 构建 RAG 代理?'


Retrieved Documents
--------------------------------------------------------------------------------

1. Source: https://python.langchain.com/docs/use_cases/question_answering/
   Type: local_text_download
   Date: 2026-05-09
   Content: Build a RAG agent with LangChain - Docs by LangChain
Skip to main content
Join us May 13th & May 14th at Interrupt, the Agent Conference by LangChain....

2. Source: https://python.langchain.com/docs/use_cases/chatbots/
   Type: local_text_download
   Date: 2026-05-09
   Content: Build a RAG agent with LangChain - Docs by LangChain
Skip to main content
Join us May 13th & May 14th at Interrupt, the Agent Conference by LangChain....

3. Source: https://python.langchain.com/docs/use_cases/question_answering/
   Type: local_text_download
   Date: 2026-05-09
   Content: prompt injection
.
​
Next steps
Now that we’ve implemented a simple RAG application via
create_agent
, we can easily incorporate new features and 

## 总结

在本 notebook 中,我们:

- 将源文档下载到 `data/source_docs/langchain/`
- 将本地文件分割成块
- 从 ModelScope 下载 `BAAI/bge-small-en-v1.5` 到 `data/models/`
- 从本地模型目录初始化 LangChain 嵌入
- 使用 BGE 嵌入创建 FAISS 向量存储
- 保存向量存储以供后续 notebook 重用
- 验证相似度搜索正常工作

继续到 **[03_simple_rag.ipynb](03_simple_rag.ipynb)** 在此向量存储之上构建 RAG 管道。